# 01 — Copy

## Background

GPU memory bandwidth is one of the core bottlenecks in deep learning training and inference. Modern GPUs (e.g., H100) offer up to 3.35 TB/s of HBM bandwidth, but only with sufficient parallelism can this bandwidth be fully utilised.

Memory copy is the most fundamental GPU operation and the ideal starting point for understanding GPU programming:
- **Block**: An independent unit of work executed in parallel on the GPU
- **Thread**: The finest-grained execution unit within a block; each thread processes a portion of the data

## Problem Definition

Copy all elements of tensor `A` into tensor `B`:

$$B[i] = A[i], \quad i = 0, 1, \ldots, N-1$$

TileLang demonstrates three implementations with progressively better performance:

| Implementation | Blocks | Threads/Block | Description |
|---------------|--------|--------------|-------------|
| Serial | 1 | 1 | Single-thread sequential |
| Multi-thread | 1 | 256 | 256 threads within one block |
| Multi-block | N/BLOCK_N | 256 | Full parallelism across blocks and threads |

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
N = 1024 * 256
A = torch.randn(N, dtype=torch.float16, device="cuda")

def ref_copy(A: torch.Tensor) -> torch.Tensor:
    return A.clone()

B = ref_copy(A)
assert torch.allclose(A, B)
print(f"Shape: {A.shape}, dtype: {A.dtype}")
print("PyTorch reference correct ✅")

## Triton Implementation

Triton uses `@triton.jit`, `tl.program_id` to get the block index, `tl.arange` to generate element offsets, and `tl.load`/`tl.store` for memory access with `mask` for boundary handling.

In [ ]:
@triton.jit
def _copy_kernel(A_ptr, B_ptr, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    tl.store(B_ptr + offs, tl.load(A_ptr + offs, mask=mask), mask=mask)

def triton_copy(A: torch.Tensor) -> torch.Tensor:
    B = torch.empty_like(A)
    N = A.numel()
    BLOCK = 1024
    _copy_kernel[(triton.cdiv(N, BLOCK),)](A, B, N, BLOCK=BLOCK)
    return B

B_tri = triton_copy(A)
print("Triton correctness:", "✅ PASSED" if torch.allclose(A, B_tri) else "❌ FAILED")

## TileLang Implementation

TileLang uses `@tilelang.jit`, `T.Kernel` to declare the grid, and `T.copy` for automatically parallelised and vectorised memory copy.

In [ ]:
# Serial (1 block × 1 thread)
@tilelang.jit
def tl_copy_serial(A):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(1, threads=1):
        T.copy(A, B)
    return B

# Multi-thread (1 block × 256 threads)
@tilelang.jit
def tl_copy_multithread(A):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(1, threads=256):
        T.copy(A, B)
    return B

# Multi-block (N//BLOCK_N blocks × 256 threads)
@tilelang.jit
def tl_copy_parallel(A, BLOCK_N: int):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(N // BLOCK_N, threads=256) as pid:
        T.copy(A[pid * BLOCK_N:(pid + 1) * BLOCK_N],
               B[pid * BLOCK_N:(pid + 1) * BLOCK_N])
    return B

def check(name, tl_fn, hyper):
    k = tl_fn.compile(**hyper)
    ok = torch.allclose(A, k(A.clone()))
    print(f"  {name}: {'✅ PASSED' if ok else '❌ FAILED'}")

check("Serial      ", tl_copy_serial,     {"N": N})
check("Multi-thread", tl_copy_multithread, {"N": N})
check("Multi-block ", tl_copy_parallel,   {"N": N, "BLOCK_N": 1024})

## Performance Comparison

Compare PyTorch, Triton, and three TileLang variants to show how parallelism improves bandwidth utilisation.

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

k_serial = tl_copy_serial.compile(N=N)
k_mt     = tl_copy_multithread.compile(N=N)
k_par    = tl_copy_parallel.compile(N=N, BLOCK_N=1024)
inp = A.clone()

results = {
    "PyTorch": bench(ref_copy, [inp]),
    "Triton":  bench(triton_copy, [inp]),
    "TileLang\nSerial":       bench(k_serial, [inp]),
    "TileLang\nMulti-thread": bench(k_mt,    [inp]),
    "TileLang\nMulti-block":  bench(k_par,   [inp]),
}

for name, ms in results.items():
    bw = N * 2 * 2 / ms / 1e9   # read + write, float16
    print(f"  {name.replace(chr(10),' '):24s}: {ms:.4f} ms  ({bw:.2f} TB/s)")

colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52", "#8172B2"]
labels = list(results.keys())
values = list(results.values())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bars = ax.bar(labels, values, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("Latency (ms)"); ax.set_title(f"Copy Latency  (N={N:,}, fp16)")
ax.set_ylim(0, max(values)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bws = [N*2*2/ms/1e9 for ms in values]
bars = ax.bar(labels, bws, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bws):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(bws)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("Effective Bandwidth (TB/s)"); ax.set_title(f"Copy Bandwidth  (N={N:,}, fp16)")
ax.set_ylim(0, max(bws)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()